In [4]:
from pyspark.sql import SparkSession
from schema import transaction_schema
from pyspark.sql import Window, functions as F

spark = SparkSession.builder \
    .appName("FullKafkaBatchProcessing") \
     .config(
                "spark.jars.packages", 
                "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"
            ) \
    .getOrCreate()                                                                                                                    

In [5]:
df = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "transaction-stream") \
    .option("startingOffsets", "earliest") \
    .option("endingOffsets", "latest") \
    .load()

print(df.count())

312714


In [ ]:
string_df = (
    df.select(
        F.from_json(
            F.col("value").cast("string"),
            transaction_schema
        ).alias("data")
    ).select("datqa.*")
)

print(string_df.schema)
print(string_df.show(3, truncate=False))

StructType([StructField('from_json(CAST(value AS STRING))', StructType([StructField('trans_date_trans_time', StringType(), True), StructField('cc_num', StringType(), True), StructField('merchant', StringType(), True), StructField('category', StringType(), True), StructField('amt', StringType(), True), StructField('first', StringType(), True), StructField('last', StringType(), True), StructField('gender', StringType(), True), StructField('street', StringType(), True), StructField('city', StringType(), True), StructField('state', StringType(), True), StructField('zip', StringType(), True), StructField('lat', StringType(), True), StructField('long', StringType(), True), StructField('city_pop', StringType(), True), StructField('job', StringType(), True), StructField('dob', StringType(), True), StructField('trans_num', StringType(), True), StructField('unix_time', StringType(), True), StructField('merch_lat', StringType(), True), StructField('merch_long', StringType(), True), StructField('i

In [ ]:
window_1h = (
        Window.partitionBy("cc_num")
        .orderBy("unix_time")
        .rangeBetween(-60 * 60, -1)
    )

df_with_win = df.WithColumn("transaction_count_1_hrs", F.count('trans_num'))